In [ ]:
# 1) Importamos las librerías necesarias

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

In [ ]:
# 2) Leemos la lista de tickers desde DIM_Tickers.csv

df_tickers = pd.read_csv("DIM_Tickers.csv")
lista_tickers = df_tickers["TickerYahoo"].drop_duplicates().tolist()

print(f"Tickers a procesar: {len(lista_tickers)}")
print(lista_tickers)

In [ ]:
# 3) Determinamos el modo: carga completa o actualización incremental
#    Si el CSV ya existe, solo descargamos desde el último registro + 1 día
#    Si no existe, descargamos el historial completo (5 años)

CSV_PATH = "precios_historicos.csv"

if os.path.exists(CSV_PATH):
    df_existente = pd.read_csv(CSV_PATH, parse_dates=["Date"])
    ultima_fecha = df_existente["Date"].max()
    fecha_inicio = (ultima_fecha + timedelta(days=1)).strftime("%Y-%m-%d")
    fecha_fin = datetime.today().strftime("%Y-%m-%d")
    MODO = "incremental"
    print(f"Modo: INCREMENTAL — descargando desde {fecha_inicio} hasta {fecha_fin}")
    print(f"Registros existentes: {len(df_existente):,}")
else:
    fecha_inicio = None
    fecha_fin = None
    MODO = "completo"
    print("Modo: COMPLETO — descargando historial de 5 años")

In [ ]:
# 4) Descargamos los datos según el modo

lista_precios = []

for t in lista_tickers:
    try:
        tk = yf.Ticker(t)

        if MODO == "incremental":
            hist = tk.history(start=fecha_inicio, end=fecha_fin)
        else:
            hist = tk.history(period="5y")

        if hist.empty:
            print(f"Sin datos nuevos para {t}")
            continue

        hist = hist.reset_index()
        hist["Ticker"] = t
        hist = hist[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None).dt.date

        lista_precios.append(hist)
        print(f"OK: {t} — {len(hist)} registros nuevos")

    except Exception as e:
        print(f"Error en {t}: {e}")

print(f"\nTickers procesados: {len(lista_precios)} de {len(lista_tickers)}")

In [ ]:
# 5) Combinamos con los datos existentes (si corresponde) y guardamos

if lista_precios:
    df_nuevos = pd.concat(lista_precios, ignore_index=True)

    if MODO == "incremental":
        df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
        df_final = df_final.drop_duplicates(subset=["Date", "Ticker"]).reset_index(drop=True)
    else:
        df_final = df_nuevos

    df_final.to_csv(CSV_PATH, index=False)
    print(f"Archivo guardado: {CSV_PATH}")
    print(f"Total registros: {len(df_final):,}")
    print(f"Período: {df_final['Date'].min()} → {df_final['Date'].max()}")
else:
    print("No hay datos nuevos para agregar — CSV sin cambios")